In [1]:
import pandas as pd
from energia import config, preprocesamiento as prep
from energia.io import guardar_tabla

In [2]:
horario, diario = prep.construir_dataset_horario()
print(horario.shape, diario.shape)
horario.head(26)

(40896, 5) (1704, 2)


,demanda_kwh,precio_bolsa_cop_kwh,precio_contrato_cop_kwh,aportes_kwh,volumen_util_kwh
fecha_hora,,,,,
2022-01-01 00:00:00,1908294.77,186.57681,245.52680,109542700.0,1.331105e+10
2022-01-01 01:00:00,1898997.50,186.57681,245.47060,109542700.0,1.331105e+10
2022-01-01 02:00:00,1898374.94,186.57681,245.28809,109542700.0,1.331105e+10
2022-01-01 03:00:00,1879004.44,198.25581,245.38345,109542700.0,1.331105e+10
2022-01-01 04:00:00,1881200.96,186.57681,245.15850,109542700.0,1.331105e+10
2022-01-01 05:00:00,1889830.83,186.57681,245.22537,109542700.0,1.331105e+10
2022-01-01 06:00:00,1612076.29,185.57681,245.30950,109542700.0,1.331105e+10
2022-01-01 07:00:00,1595284.92,183.55681,244.77560,109542700.0,1.331105e+10
2022-01-01 08:00:00,1613834.34,185.57681,244.36670,109542700.0,1.331105e+10


In [3]:
prep.reporte_calidad(horario)

,nulos,ceros,negativos,min,max,media
demanda_kwh,0,0,0,1.595285e+06,3.664484e+06,2.920096e+06
precio_bolsa_cop_kwh,0,0,0,8.905597e+01,2.675650e+03,4.274293e+02
precio_contrato_cop_kwh,0,0,0,2.377350e+02,3.836601e+02,2.865167e+02
aportes_kwh,0,0,0,5.579500e+07,6.430755e+08,2.058925e+08
volumen_util_kwh,0,0,0,4.958518e+09,1.564980e+10,1.190366e+10


In [4]:
esperado = pd.date_range(horario.index.min(), horario.index.max(), freq="h")
pd.Series({
    "horas_esperadas": len(esperado),
    "horas_presentes": len(horario),
    "horas_faltantes": len(esperado.difference(horario.index)),
    "indices_duplicados": int(horario.index.duplicated().sum()),
    "dias_incompletos": int((horario.notna().all(axis=1).groupby(horario.index.date).sum() < 24).sum()),
})

horas_esperadas       40896
horas_presentes       40896
horas_faltantes           0
indices_duplicados        0
dias_incompletos          0
dtype: int64

In [5]:
pd.Series({col: horario[col].groupby(horario.index.date).nunique().max() for col in config.EXOGENAS_DIARIAS},
          name="valores_distintos_por_dia")

aportes_kwh         1
volumen_util_kwh    1
Name: valores_distintos_por_dia, dtype: int64

In [6]:
prep.validar(horario)
guardar_tabla(horario, config.ARCHIVO_HORARIO)

WindowsPath('C:/Users/57310/Desktop/pronostico-energia-nr/pronostico-energia-nr/data/interim/dataset_horario.parquet')